In [ ]:
import xarray as xr
import xrtoy
from dask.diagnostics import (
    ProgressBar,
    Profiler,
    CacheProfiler,
    ResourceProfiler,
    visualize,
)

In [ ]:
# Generate data
frame_shape = (80, 324, 244)
filenames = xrtoy.save_data(
    "/home/drscience/tmp/mydata",
    frame_shape,
    200,
    n_files=20,
    mode="w",
)
print(filenames)

In [ ]:
ds = xr.open_mfdataset(filenames, engine="xrtbasic")
ds

In [ ]:
mean_ds = ds.mean(("time", "coord0"))
lazy_da = mean_ds["myvar"]

with Profiler() as prof, ResourceProfiler(dt=0.25) as rprof, CacheProfiler() as cprof:
    lazy_da.compute()

visualize([prof, rprof, cprof])

In [ ]:
mean_ds = ds.mean("coord0")
mean_da = mean_ds["myvar"]
reversed_mean_da = mean_da.reindex(time=list(reversed(mean_da.time)))

with Profiler() as prof, ResourceProfiler() as rprof, CacheProfiler() as cprof:
    xr.cov(mean_da, -reversed_mean_da, dim="time").compute()

visualize([prof, rprof, cprof])